## Data Preprocessing Script

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import pickle
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [29]:
df = pd.read_csv('adult.csv')

# Replace "?" with NaN
df = df.replace(' ?', np.nan)
df = df.replace('?', np.nan)

# Drop rows with missing values (or you can impute)
df = df.dropna()

In [30]:
# Separate features and target
X = df.drop('income', axis=1)
y = df['income'].map({'<=50K': 0, '>50K': 1})

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Class Distribution:\n{y.value_counts()}")

Features: 14
Samples: 30162
Class Distribution:
income
0    22654
1     7508
Name: count, dtype: int64


/var/folders/tt/4lzq7_dx6637wp_xjqqf6wm80000gn/T/ipykernel_2166/1738691243.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


In [31]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Save test data for Streamlit
X_test.to_csv('test_data.csv', index=False)
print("\nTest data saved as 'test_data.csv'")


Training set: 24129 samples
Test set: 6033 samples

Test data saved as 'test_data.csv'


## Train All Models

In [32]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols)
    ])

In [33]:
#define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

In [34]:
# Create directory for models
os.makedirs('models', exist_ok=True)

In [35]:
# Store results
results = []
trained_models = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Create pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline.named_steps['classifier'], 'predict_proba') else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    # Store results
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc
    })
    
    # Save model
    with open(f'models/{name.replace(" ", "_")}.pkl', 'wb') as f:
        pickle.dump(pipeline, f)
    
    trained_models[name] = pipeline
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  AUC: {auc:.4f}" if auc else "  AUC: N/A")

# Display results
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("Model Performance Summary:")
print("="*80)
print(results_df.to_string(index=False))


Training Logistic Regression...
  Accuracy: 0.8543
  AUC: 0.9136

Training Decision Tree...
  Accuracy: 0.8158
  AUC: 0.7537

Training KNN...
  Accuracy: 0.8341
  AUC: 0.8672

Training Naive Bayes...
  Accuracy: 0.6010
  AUC: 0.8300

Training Random Forest...
  Accuracy: 0.8556
  AUC: 0.9114

Model Performance Summary:
              Model  Accuracy      AUC  Precision   Recall       F1      MCC
Logistic Regression  0.854301 0.913589   0.750201 0.621838 0.680015 0.591088
      Decision Tree  0.815846 0.753669   0.630247 0.629827 0.630037 0.507450
                KNN  0.834079 0.867233   0.683248 0.621838 0.651098 0.543609
        Naive Bayes  0.601028 0.830016   0.379494 0.948735 0.542134 0.387561
      Random Forest  0.855627 0.911438   0.746677 0.635819 0.686803 0.597016
